<a href="https://colab.research.google.com/github/ronronrivera/deep_learning-notebooks/blob/main/rice_type_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install the Kaggle library
!pip install kaggle --upgrade --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 9.8 MB/s eta 0:00:00


In [2]:
import os

print(os.listdir('/kaggle/input/datasets/mssmartypants/rice-type-classification'))

['riceClassification.csv']


In [3]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
data_df = pd.read_csv('/kaggle/input/datasets/mssmartypants/rice-type-classification/riceClassification.csv')

data_df.head()

,id,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,1,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1
3,4,3073,77.033628,51.928487,0.738639,3157,62.551300,0.783529,210.657,0.870203,1.483456,1
4,5,3693,85.124785,56.374021,0.749282,3802,68.571668,0.769375,230.332,0.874743,1.510000,1


In [5]:
#drop rows with any missing values
data_df.dropna(inplace=True)

#drop id from datasets since it won't matter in training a model
data_df.drop(['id'], axis=1, inplace=True)

print(data_df.shape)

(18185, 11)


In [6]:
data_df.head()

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1
3,3073,77.033628,51.928487,0.738639,3157,62.551300,0.783529,210.657,0.870203,1.483456,1
4,3693,85.124785,56.374021,0.749282,3802,68.571668,0.769375,230.332,0.874743,1.510000,1


In [7]:
print(data_df['Class'].unique())

[1 0]


In [8]:
print(data_df['Class'].value_counts())

Class
1    9985
0    8200
Name: count, dtype: int64


In [9]:
original_df = data_df.copy()

#normalize data from 0 to 1
for column in data_df.columns:
  data_df[column] = data_df[column]/data_df[column].abs().max()

In [10]:
data_df.head()

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,0.444368,0.503404,0.775435,0.744658,0.424873,0.666610,0.741661,0.537029,0.844997,0.368316,1.0
1,0.281293,0.407681,0.622653,0.750489,0.273892,0.530370,0.804230,0.409661,0.919215,0.371471,1.0
2,0.298531,0.416421,0.630442,0.756341,0.284520,0.546380,0.856278,0.412994,0.959862,0.374747,1.0
3,0.300979,0.420463,0.629049,0.764024,0.286791,0.548616,0.883772,0.414262,0.961818,0.379222,1.0
4,0.361704,0.464626,0.682901,0.775033,0.345385,0.601418,0.867808,0.452954,0.966836,0.386007,1.0


In [11]:
X = np.array(data_df.iloc[:, :-1])
Y = np.array(data_df.iloc[:, -1])

In [12]:
#separate data into 2 for trainin and testing

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3)

In [13]:
#separate data into 2 for testing and validation

X_test, X_val, Y_test, Y_val = train_test_split(X_test, Y_test, test_size=0.5)


In [14]:
class dataset(Dataset):
  def __init__(self, X, Y):
    self.X = torch.tensor(X, dtype=torch.float32).to(device)
    self.Y = torch.tensor(Y, dtype=torch.float32).to(device)

  def __len__(self):
    return len(self.X)

  def __getitem__(self, index):
    return self.X[index], self.Y[index]



In [15]:
training_data = dataset(X_train, Y_train)
validation_data = dataset(X_val, Y_val)
testing_data = dataset(X_test, Y_test)

In [16]:
train_dataloader = DataLoader(training_data, batch_size=8, shuffle=True)
validation_dataloader = DataLoader(validation_data, batch_size=8, shuffle=True)
testing_dataloader = DataLoader(testing_data, batch_size=8, shuffle=True)




In [17]:
HIDDEN_NEURONS = 10

class MyModel(nn.Module):

  def __init__(self):
    super(MyModel, self).__init__()
    self.input_layer = nn.Linear(X.shape[1], HIDDEN_NEURONS)
    self.linear = nn.Linear(HIDDEN_NEURONS, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    x = self.input_layer(x)
    x = self.linear(x)
    x = self.sigmoid(x)
    return x


In [18]:
model = MyModel().to(device)

summary(model, (X.shape[1],))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 10]             110
            Linear-2                    [-1, 1]              11
           Sigmoid-3                    [-1, 1]               0
Total params: 121
Trainable params: 121
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------


In [19]:
criterion = nn.BCELoss()
optimizer = Adam(model.parameters(), lr=1e-3)

In [20]:
total_loss_train_plot = []
total_loss_validation_plot = []

total_accuracy_train_plot = []
total_accuracy_validation_plot = []

epochs = 10

for epoch in range(epochs):
  total_acc_train = 0
  total_loss_train = 0

  total_acc_val = 0
  total_loss_val = 0

  for data in train_dataloader:
    inputs, labels = data

    prediction = model(inputs).squeeze(1)
    batch_loss = criterion(prediction, labels)

    total_loss_train += batch_loss.item()

    acc = (prediction.round() == labels).sum().item()

    total_acc_train += acc
    batch_loss.backward()
    optimizer.step()
    optimizer.zero_grad()

  with torch.no_grad():
      for data in validation_dataloader:
          inputs, label = data
          
          prediction = model(inputs).squeeze(1)
          batch_loss = criterion(prediction, label)

          total_loss_val += batch_loss.item()
          acc = (prediction.round() == label).sum().item()

          total_acc_val += acc
          

  total_loss_train_plot.append(round(total_loss_train/1000, 4))
  total_loss_validation_plot.append(round(total_loss_val/1000, 4))
    
  total_accuracy_train_plot.append(round(total_acc_train/training_data.__len__() * 100, 4))  
  total_accuracy_validation_plot.append(round(total_acc_val/validation_data.__len__() * 100, 4))  

  print(f'''
          Epoch No. {epoch+1}, Train Loss: {round(total_loss_train/1000, 4)}, Train Accuracy: {round(total_acc_train/training_data.__len__() * 100, 4)}
                               Validation Loss: {round(total_loss_val/1000, 4)}, Validation Accuracy: {round(total_acc_val/validation_data.__len__() * 100, 4)}     
      ''')
  print('='*100)



          Epoch No. 1, Train Loss: 0.6416, Train Accuracy: 86.684
                               Validation Loss: 0.044, Validation Accuracy: 98.4604     
      

          Epoch No. 2, Train Loss: 0.1293, Train Accuracy: 98.4838
                               Validation Loss: 0.0218, Validation Accuracy: 98.1305     
      

          Epoch No. 3, Train Loss: 0.0817, Train Accuracy: 98.5623
                               Validation Loss: 0.0184, Validation Accuracy: 98.3871     
      

          Epoch No. 4, Train Loss: 0.0705, Train Accuracy: 98.6566
                               Validation Loss: 0.0167, Validation Accuracy: 98.5704     
      

          Epoch No. 5, Train Loss: 0.0667, Train Accuracy: 98.5702
                               Validation Loss: 0.0176, Validation Accuracy: 98.4971     
      

          Epoch No. 6, Train Loss: 0.0651, Train Accuracy: 98.6095
                               Validation Loss: 0.0164, Validation Accuracy: 98.5704     
      

          E